<a href="https://colab.research.google.com/github/KULDEEPSONI-source/MACHINE-LEARNING/blob/main/HyperparameterOptimization(HPO).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

----
----
----
----

**Hyperparameter Optimization (HPO)** is the process of finding the optimal configuration of "settings" for a machine learning model to maximize its performance.

Think of it this way:

* **Parameters** are the internal variables that a model *learns* on its own during training (e.g., weights in a Logistic Regression).
* **Hyperparameters** are the settings *you* decide before training begins (e.g., the learning rate, the depth of a tree, or the type of penalty in a regression).

### Why it matters

The "default" settings of an algorithm are rarely the best for your specific data. HPO allows you to fine-tune the model to prevent underfitting (too simple) or overfitting (too complex).

---

### Common HPO Strategies

#### 1. Grid Search

* **How it works:** You define a list of values for each hyperparameter. The algorithm tries **every possible combination**.
* **Pros:** Thorough; guaranteed to find the best combination within your list.
* **Cons:** Extremely slow. As you add more hyperparameters, the "grid" grows exponentially (the "Curse of Dimensionality").

#### 2. Random Search

* **How it works:** Instead of trying every point, it picks random combinations from the defined range.
* **Pros:** Surprisingly more efficient than Grid Search. It often finds a "good enough" solution much faster because it explores a wider space of parameters.
* **Cons:** No guarantee it will find the absolute "global" optimum.

#### 3. Bayesian Optimization

* **How it works:** This is the "smart" approach. It uses past results to build a probability model of the objective function, predicting which parameters will likely yield the best results next.
* **Pros:** Much faster than Grid or Random Search; focuses on the most promising areas of the hyperparameter space.

---

### Implementation Example (using `GridSearchCV`)

In `scikit-learn`, we use `GridSearchCV` to combine cross-validation with grid search:

```python
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Define the hyperparameters to test
param_grid = {
    'C': [0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs']
}

# Initialize model and search
model = LogisticRegression()
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')

# Fit to training data
grid_search.fit(X_train, y_train)

# View best results
print("Best Parameters:", grid_search.best_params_)

```

---

### Key Considerations

* **Cross-Validation:** Always pair HPO with cross-validation (`cv=5`). If you tune your hyperparameters on the *test* set, you are committing a cardinal sin of machine learning—**data leakage**—and your results will be falsely optimistic.
* **Search Space:** Start broad and narrow down. If your grid search returns an edge value (e.g., the maximum value you provided), re-run the search with a higher range.
* **Early Stopping:** For complex models (like XGBoost or Neural Networks), use "early stopping" to halt training if the model stops improving, saving you significant time.

Are you currently working on a specific model, such as a Random Forest or a Neural Network, for which you need to tune hyperparameters?
---
---
---
----
----


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score

In [2]:
dataset = pd.read_csv('/content/breast_cancer.csv')

In [3]:
dataset.head(5)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
# Distribution of y-variable
dataset.diagnosis.value_counts()

,count
diagnosis,
B,357
M,212


In [5]:
dataset.diagnosis.value_counts()/len(dataset)*100

,count
diagnosis,
B,62.741652
M,37.258348


In [6]:
dataset.shape

(569, 33)

In [7]:
dataset['diagnosis'] = dataset['diagnosis'].map({'M': 1, 'B': 0})

In [8]:
dataset.head(5)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,1,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,1,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,1,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,1,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [9]:
dataset.diagnosis.value_counts()

,count
diagnosis,
0,357
1,212


In [10]:
# X & y

X = dataset.iloc[:, [2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]]
y = dataset.diagnosis.values

#### Train Test Split


In [11]:
# Train-Test split

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


In [12]:
len(X_train)

455

In [13]:
len(X_test)

114

#### Feature Scaling

In [14]:
# Standardization
# Normalization
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

#### Model Building (RF):

In [15]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [16]:
# Predicting the test set values

y_pred = model.predict(X_test)

In [17]:
# Metrics

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
result = accuracy_score(y_test, y_pred)
print("Accuracy of the base RF model is: ", round(result*100,2))

Accuracy of the base RF model is:  93.86


In [18]:
cm = confusion_matrix(y_test, y_pred)
print("CM is: ")
print(cm)

CM is: 
[[63  4]
 [ 3 44]]


In [19]:
cr = classification_report(y_test, y_pred)
print("Classification Report is: ")
print(cr)

Classification Report is: 
              precision    recall  f1-score   support

           0       0.95      0.94      0.95        67
           1       0.92      0.94      0.93        47

    accuracy                           0.94       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.94      0.94      0.94       114



### Manual HPO

In [20]:
n_estimators_list = [1,2,3,10,50,100,200]

for estim_list in n_estimators_list:
  model = RandomForestClassifier(n_estimators=estim_list)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  result = accuracy_score(y_test, y_pred)
  print("\n Estimator Value", estim_list)
  print("Accuracy is", result)


 Estimator Value 1
Accuracy is 0.9035087719298246

 Estimator Value 2
Accuracy is 0.8859649122807017

 Estimator Value 3
Accuracy is 0.9122807017543859

 Estimator Value 10
Accuracy is 0.9385964912280702

 Estimator Value 50
Accuracy is 0.9385964912280702

 Estimator Value 100
Accuracy is 0.9298245614035088

 Estimator Value 200
Accuracy is 0.956140350877193


In [21]:
leaf_size = [1,2,3,4,5,10]

for i in leaf_size:
  model = RandomForestClassifier(n_estimators=10, min_samples_leaf=i)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  result = accuracy_score(y_test, y_pred)
  print("\n Estimator Value", i)
  print("Accuracy is", result)



 Estimator Value 1
Accuracy is 0.9473684210526315

 Estimator Value 2
Accuracy is 0.9298245614035088

 Estimator Value 3
Accuracy is 0.9385964912280702

 Estimator Value 4
Accuracy is 0.9298245614035088

 Estimator Value 5
Accuracy is 0.9385964912280702

 Estimator Value 10
Accuracy is 0.9210526315789473


### Random Search CV

In [22]:
n_estimators = [int(x) for x in np.linspace(start=100, stop=1000, num=10)]
max_depth = [int(x) for x in np.linspace(start=10, stop=110, num=11)]

print(n_estimators)
print(max_depth)

[100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
[10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110]


In [23]:
from sklearn.model_selection import RandomizedSearchCV

n_estimators = [int(x) for x in np.linspace(start=100, stop=1000, num=10)]
max_depth = [int(x) for x in np.linspace(start=10, stop=110, num=11)]
min_samples_leaf = [1,2,4,10,20,50,100]
min_samples_split = [2,3,4,5,8,10,20,50,100,200]
bootstrap = [True, False]

# Create the random grid

random_grid = {'n_estimators': n_estimators,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'bootstrap': bootstrap,
               'min_samples_leaf': min_samples_leaf}

On each iteration, the algorithm will choose a different combination of the features. Altogether, there are 15,400 combinations. However the benefit of random search is that we are not trying every combination, but selecting at random to sample a wide range of values

In [24]:
# Use the random grid to search for best hyper parameters
# First create the base model to tune

rf = RandomForestClassifier()
# Random search of parameters using 3 fold cross validations
rf_random = RandomizedSearchCV(estimator=rf, param_distributions=random_grid, n_iter=100, cv=3, n_jobs=-1)

# Fit the random search model
rf_random.fit(X_train, y_train)

RandomizedSearchCV(cv=3, estimator=RandomForestClassifier(), n_iter=100,
                   n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [10, 20, 30, 40, 50, 60,
                                                      70, 80, 90, 100, 110],
                                        'min_samples_leaf': [1, 2, 4, 10, 20,
                                                             50, 100],
                                        'min_samples_split': [2, 3, 4, 5, 8, 10,
                                                              20, 50, 100,
                                                              200],
                                        'n_estimators': [100, 200, 300, 400,
                                                         500, 600, 700, 800,
                                                         900, 1000]})

In [25]:
def evaluate(model, test_features, test_labels):
  predictions = model.predict(test_features)
  accuracy = accuracy_score(test_labels, predictions)
  print('Model Performance')
  print('Accuracy = {:0.2f}%.'.format(accuracy))
  return accuracy

In [26]:
base_model = RandomForestClassifier(n_estimators=5, random_state=42)
base_model.fit(X_train, y_train)
base_accuracy = evaluate(base_model,X_test,y_test)

Model Performance
Accuracy = 0.93%.


In [27]:
best_random = rf_random.best_estimator_
print(best_random)

RandomForestClassifier(max_depth=80, n_estimators=1000)


In [28]:
random_accuracy = evaluate(best_random,X_test,y_test)

Model Performance
Accuracy = 0.94%.


In [29]:
print('Improvement of {:0.2f}%.'.format(100*(random_accuracy-base_accuracy)/base_accuracy))

Improvement of 0.94%.


### Grid Search CV


You have set up a `GridSearchCV` to perform an exhaustive search for the best hyperparameters for your `RandomForestClassifier`. Since `GridSearchCV` evaluates every single combination of the provided parameters, it is the most thorough way to fine-tune your model.

Here is the breakdown of the parameters you are using:

### 1. `param_grid` (The Search Space)

This dictionary defines the specific values the algorithm will test:

* **`bootstrap`**: Whether bootstrap samples (sampling with replacement) are used when building trees.
* **`max_depth`**: The maximum depth of each tree. Limiting this prevents trees from growing too complex (overfitting).
* **`max_features`**: The number of features to consider when looking for the best split at each node.
* **`min_samples_leaf`**: The minimum number of samples required to be at a leaf node. Higher numbers smooth the model and reduce overfitting.
* **`min_samples_split`**: The minimum number of samples required to split an internal node.
* **`n_estimators`**: The number of trees in the forest. Generally, more trees provide better performance but increase computation time.

---

### 2. `GridSearchCV` Constructor Parameters

| Parameter | Description |
| --- | --- |
| **`estimator`** | The model object you want to tune (your `RandomForestClassifier`). |
| **`param_grid`** | The dictionary of parameters you defined above. The grid search will build a model for every possible combination (Cartesian product) of these values. |
| **`cv`** | Stands for "Cross-Validation." Setting it to `3` means the dataset will be split into 3 parts (folds). The model will train on 2 and validate on 1, repeating the process 3 times for each parameter combination to ensure the results are robust. |
| **`n_jobs`** | Controls parallel processing. Setting it to `-1` tells Scikit-Learn to use **all available CPU cores** on your machine. This significantly speeds up the search. |
| **`verbose`** | Controls how much information is printed to the console while the search is running. A value of `2` provides output about the progress of each fit. |

---

### Important Considerations for your Setup

* **Computational Cost:** You have a massive grid. To calculate the number of combinations:

$$1 (\text{bootstrap}) \times 6 (\text{depth}) \times 4 (\text{features}) \times 3 (\text{leaf}) \times 3 (\text{split}) \times 8 (\text{estimators}) = 1,728 \text{ combinations}$$



With `cv=3`, the model will be trained **5,184 times**. Ensure you have enough time and memory for this!
* **`verbose=2`:** Since you have so many combinations, your console will be flooded with logs. If it's too much, you can reduce this to `1`.
* **`refit=True` (Default):** By default, `GridSearchCV` will automatically retrain the model on the *entire* training dataset using the best hyperparameters found during the search, so `grid_search` can be used directly for predictions immediately after `fit`.
---
---
---
---


In [30]:
from sklearn.model_selection import GridSearchCV
# Create the parameter grid based on the results of random search
param_grid = {
    'bootstrap': [True],
    'max_depth': [75, 80, 85, 90, 95, 100],
    'max_features': [2, 3, 4, 5],
    'min_samples_leaf': [1, 2, 3],
    'min_samples_split': [8, 10, 12],
    'n_estimators': [200, 250, 270, 300, 350, 400, 450, 500]
}
# Create a based model
rf_gd = RandomForestClassifier()
# Instantiate the grid search model
grid_search = GridSearchCV(estimator = rf_gd, param_grid = param_grid,
                          cv = 3, n_jobs = -1, verbose = 2)

In [ ]:
# Fit the grid search to the data
grid_search.fit(X_train, y_train)
grid_search.best_params_

Fitting 3 folds for each of 1728 candidates, totalling 5184 fits


In [ ]:
best_grid = grid_search.best_estimator_
grid_accuracy = evaluate(best_grid, X_test, y_test)

In [ ]:
print('Improvement of {:0.2f}%.'.format(100*(grid_accuracy-base_accuracy)/base_accuracy))

In [ ]:
best_grid